> **Historical notebook:** ผลรุ่นแรก ไม่ใช่รุ่นอ้างอิงปัจจุบัน ดู [model/README.md](model/README.md)

# LEPTOLIC / INT182 รอบ 3 — Repeated nested CV

รันกับ ZIP NHEFS จริงสำเร็จแล้ว ดูผลและข้อจำกัดใน `round3/results/nhefs_20261105/README_TH.md`
Legacy AUC 0.6761 → ใหม่ 0.7054; Brier 0.05653 → 0.05579
แต่ paired Brier 95% CI คร่อมศูนย์ จึงยังไม่ผ่าน primary improvement criterion
Test เคยเปิดเผยแล้ว ไม่ใช่ external validation หรือการยืนยันกับคนไทย

อ่าน `round3/CV_PROTOCOL_TH.md` ก่อนรันซ้ำ ใช้ checkout เต็มและ ZIP ของผู้ใช้
ระดับ 1–3 เป็นช่วงคะแนน development OOF ไม่ใช่เกณฑ์คลินิก

In [ ]:
%pip install -r round3/requirements.txt

## ตรวจหลักฐานและ protocol ก่อนฝึก
ไม่ใช้ผล Test เลือกฟีเจอร์/threshold และไม่อ้างว่า Test เดิมเป็นข้อมูลใหม่ที่ไม่เคยเห็น

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown
display(Markdown(Path('round3/CV_PROTOCOL_TH.md').read_text(encoding='utf-8')))

## ฝึกด้วยข้อมูลจริง
เลือก work directory ใหม่สำหรับแต่ละ attempt และกำหนด DATA_ZIP ให้ชี้ ZIP ที่มีข้อมูล/layouts ครบ 8 ไฟล์
เมื่อใช้ ZIP จะไม่ดาวน์โหลด CDC การตรวจรูปแบบไม่ใช่การยืนยันแหล่งกำเนิดโดยอิสระ
ดู input manifest และจำนวนคนที่ถูกคัดออก และยืนยัน codebook ก่อนรับรอง target

In [ ]:
import os
from round3.cv_study import run_study
# Set these environment variables or edit the defaults before a new run.
WORK = Path(os.environ.get('LEPTOLIC_WORK_DIR', 'round3_work/nested_zip_01')).resolve()
DATA_ZIP = Path(os.environ.get('LEPTOLIC_DATA_ZIP', 'NHEFS_compressed.zip'))
if not DATA_ZIP.is_file():
    raise FileNotFoundError('Set DATA_ZIP to your eight-file NHEFS ZIP before running')
results = run_study(WORK, data_zip=DATA_ZIP)

## อ่านผลและระดับ
ตรวจจำนวนคน/เหตุการณ์และช่วงความไม่แน่นอนของทุกระดับ ระดับสูงอาจมี observed rate ไม่สูงขึ้นใน Test ขนาดเล็ก ต้องรายงานตามจริง

In [ ]:
print(results['test_new'])
print(results['test_legacy_with_same_threshold_policy'])
print(results['paired_test_comparison'])
print(results['test_levels'])
display(Markdown((WORK/'report_TH.md').read_text(encoding='utf-8')))

## ทดลองกรอกข้อมูล
ตัวอย่างด้านล่างอาจถูกปฏิเสธถ้าอยู่นอก training range; ไม่เหมาะกับผู้เคยได้รับการวินิจฉัยแล้ว
ไม่ใช่เครื่องมือวินิจฉัย/เลือกการรักษา

In [ ]:
from round3.predict import predict_file
INPUT = dict(age_years=50, sex_female=0, baseline_sbp=120, baseline_dbp=75, sleep_hours=7)
answer = predict_file(WORK/'model.json', INPUT, previously_diagnosed=False)
print(answer)

## ไฟล์ผล
เมื่อรันข้อมูลจริงสำเร็จจึงมี report, metrics, model และ predictions ไม่ใช้ผล software fixtures แทนผลจริง

In [ ]:
from IPython.display import FileLink
for name in ['report_TH.md', 'metrics.json', 'model.json', 'test_predictions.csv', 'development_oof.csv']:
    display(FileLink(str(WORK/name)))